In [3]:
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import random

# ============================================================
# CONFIGURATION (change paths if needed)
# ============================================================
MODELS_ROOT = Path(r"D:\mtech\sem 4\gen Ai\experiment\euro\eurosat_output\models\best")
REAL_ROOT = Path(r"D:\mtech\sem 4\gen Ai\project\dataset\euro_large\EuroSAT_RGB")
OUT_DIR = Path("generated_from_models")
OUT_DIR.mkdir(exist_ok=True)

LATENT_DIM = 100
NGF = 64
NUM_CHANNELS = 3
IMAGE_SIZE = 64

NUM_IMAGES = 10      # 2x5 grid
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Losses and their generator file names (adjust if your files have different names)
LOSS_INFO = {
    "standard": "standard_G_best.pth",
    "lsgan": "Isgan_G_best.pth",   # note capital I
    "wgan": "wgan_G_best.pth",
    "wgangp": "wgangp_G_best.pth",
    "hinge": "hinge_G_best.pth"
}
PRETTY_NAMES = {
    "standard": "Standard GAN",
    "lsgan": "LSGAN",
    "wgan": "WGAN",
    "wgangp": "WGAN-GP",
    "hinge": "Hinge Loss"
}

# ============================================================
# 1. GENERATOR ARCHITECTURE (same as training)
# ============================================================
class Generator(torch.nn.Module):
    def __init__(self, ngf=64, latent_dim=100, nc=3):
        super(Generator, self).__init__()
        self.main = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            torch.nn.BatchNorm2d(ngf * 8),
            torch.nn.ReLU(True),
            torch.nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(ngf * 4),
            torch.nn.ReLU(True),
            torch.nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(ngf * 2),
            torch.nn.ReLU(True),
            torch.nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(ngf),
            torch.nn.ReLU(True),
            torch.nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, z):
        return self.main(z)

# ============================================================
# 2. LOAD REAL IMAGES (optional)
# ============================================================
def load_real_images():
    if not REAL_ROOT.exists():
        print(f"WARNING: REAL_ROOT folder not found at {REAL_ROOT}")
        return [], []
    class_folders = sorted([d for d in REAL_ROOT.iterdir() if d.is_dir()])
    real_imgs = []
    class_names = []
    for cf in class_folders[:10]:
        imgs = list(cf.glob("*.jpg")) + list(cf.glob("*.png"))
        if not imgs:
            continue
        img = Image.open(random.choice(imgs)).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        real_imgs.append(np.array(img))
        class_names.append(cf.name)
    if len(real_imgs) == 0:
        print("WARNING: No real images found. Check that class folders contain .jpg or .png files.")
    else:
        print(f"Loaded {len(real_imgs)} real images from {len(class_names)} classes.")
    return real_imgs, class_names

real_imgs, class_names = load_real_images()

# ============================================================
# 3. GENERATE GRID AND INDIVIDUAL SAMPLES FOR EACH LOSS
# ============================================================
def generate_images(generator, num_images, latent_dim, device):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(num_images, latent_dim, 1, 1, device=device)
        fake = generator(z).detach().cpu()
        fake = (fake + 1) / 2   # denormalize
    return fake

for loss_name, file_name in LOSS_INFO.items():
    model_path = MODELS_ROOT / file_name
    if not model_path.exists():
        print(f"Model not found: {model_path}")
        continue

    generator = Generator(ngf=NGF, latent_dim=LATENT_DIM, nc=NUM_CHANNELS).to(DEVICE)

    # Load state dict with key remapping (fix 'net.' -> 'main.')
    state_dict = torch.load(model_path, map_location=DEVICE)
    if 'generator_state_dict' in state_dict:
        state_dict = state_dict['generator_state_dict']

    first_key = next(iter(state_dict.keys()))
    if first_key.startswith('net.'):
        new_state_dict = {k.replace('net.', 'main.'): v for k, v in state_dict.items()}
    else:
        new_state_dict = state_dict

    generator.load_state_dict(new_state_dict)
    generator.eval()
    print(f"Loaded {loss_name} generator from {file_name}")

    # Generate images
    fake_tensors = generate_images(generator, NUM_IMAGES, LATENT_DIM, DEVICE)

    # Save individual images
    loss_dir = OUT_DIR / loss_name
    loss_dir.mkdir(exist_ok=True)
    for i, img_tensor in enumerate(fake_tensors):
        img_np = img_tensor.permute(1,2,0).numpy()
        plt.imsave(loss_dir / f"sample_{i}.png", img_np)

    # Create a 2x5 grid
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    axes = axes.flatten()
    for i in range(NUM_IMAGES):
        axes[i].imshow(fake_tensors[i].permute(1,2,0).numpy())
        axes[i].axis('off')
    plt.suptitle(f"{PRETTY_NAMES[loss_name]} – EuroSAT Generated Samples", fontsize=12)
    plt.tight_layout()
    grid_path = OUT_DIR / f"{loss_name}_grid.png"
    plt.savefig(grid_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved grid: {grid_path}")

# ============================================================
# 4. REAL VS GENERATED COMPARISON (only if we have real images)
# ============================================================
if len(real_imgs) == 0:
    print("\nSkipping real vs generated comparisons because no real images were loaded.")
else:
    N_COMPARE = min(len(real_imgs), NUM_IMAGES)
    for loss_name in LOSS_INFO.keys():
        loss_dir = OUT_DIR / loss_name
        if not loss_dir.exists():
            continue
        gen_files = sorted(loss_dir.glob("*.png"))[:N_COMPARE]
        if len(gen_files) < N_COMPARE:
            print(f"Not enough generated images for {loss_name}, need {N_COMPARE}")
            continue

        fig, axes = plt.subplots(2, N_COMPARE, figsize=(N_COMPARE * 1.2, 2.5))
        for i in range(N_COMPARE):
            # Real image
            axes[0, i].imshow(real_imgs[i])
            axes[0, i].set_title(class_names[i][:6], fontsize=6)
            axes[0, i].axis('off')
            # Generated image
            gen_img = Image.open(gen_files[i]).resize((IMAGE_SIZE, IMAGE_SIZE))
            axes[1, i].imshow(np.array(gen_img))
            axes[1, i].axis('off')
        axes[0, 0].set_ylabel("Real", fontsize=10)
        axes[1, 0].set_ylabel(PRETTY_NAMES[loss_name], fontsize=10)
        plt.suptitle(f"EuroSAT: Real vs {PRETTY_NAMES[loss_name]}", fontsize=12)
        plt.tight_layout()
        out_comp = OUT_DIR / f"real_vs_{loss_name}.png"
        plt.savefig(out_comp, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved comparison: {out_comp}")

print(f"\nAll grid and individual images saved in: {OUT_DIR.absolute()}")

Loaded standard generator from standard_G_best.pth
Saved grid: generated_from_models\standard_grid.png
Model not found: D:\mtech\sem 4\gen Ai\experiment\euro\eurosat_output\models\best\Isgan_G_best.pth
Loaded wgan generator from wgan_G_best.pth
Saved grid: generated_from_models\wgan_grid.png
Loaded wgangp generator from wgangp_G_best.pth
Saved grid: generated_from_models\wgangp_grid.png
Loaded hinge generator from hinge_G_best.pth
Saved grid: generated_from_models\hinge_grid.png

Skipping real vs generated comparisons because no real images were loaded.

All grid and individual images saved in: d:\mtech\sem 4\gen Ai\gitcode\Comparative Analysis of GAN Loss Functions\generated_from_models
